<a href="https://colab.research.google.com/github/mrmeerkat0815/Cultural-Data-analysis-Nino/blob/main/Copia_di_Week4_Workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Save your own copy first.** Unless this notebook is already in your own Google Drive, Colab does not keep your changes. Go to **File → Save a copy in Drive** and work in the copy that opens in a new tab. You will find it later in Google Drive, in *Colab Notebooks*.

> **Important:** Before you run any cells in this notebook, in the top menu go to **Runtime|Change runtime type** and switch your runtime to **Python 3** with hardware accelerator **T4 GPU**.
>
> When asked, please **allow the notebook access to Google Drive**: choose the same Google account you use for Colab and tick all the permissions. Drive is used to read the models from the shared course cache (the CDA-CACHE shortcut you set up following the instructions on Canvas).
>
> Please run the cell below to set up the environment. This takes a little while.
>
> **This workshop consists of three parts. The first part, 'Case Study: Hate speech on social media', covers hate-speech detection and fine-tuning Transformers. The second part, 'A Second Case Study: Do Two Sentences Mean the Same Thing', is about text-to-text Transformers, which are the reason why people now also use LLMs for tasks like sentiment analysis. The third part, 'The lecture again, with language models', does the steps of the lecture again with language models, without any training. What to hand in is explained in the box below the goals.**


In [ ]:
# Keep cell
# Running this takes a while!
%%capture

import sys
import os
import nltk

import matplotlib.pyplot as plt
import torch

import pandas as pd
import numpy as np

try:
  import learntools.core
  from learntools.core import binder
  binder.bind(globals())
except:
  !pip install git+https://github.com/goto4711/learntools.git
  import learntools.core
  from learntools.core import binder
  binder.bind(globals())

if not(os.path.exists('text_workshop_tests.py')):
  print('Downloading text_workshop_tests.py\n')
  !wget https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/text_workshop_tests.py


from text_workshop_tests import *


#Load models ...

# The models come from the shared course cache (the CDA-CACHE shortcut in your Google Drive),
# so that not everyone has to download them from Hugging Face at the same time.
# The shared cache is read-only for you. Everything Hugging Face needs to write
# (datasets, temporary files) therefore stays on this Colab machine.
LOCAL_HF = "/content/hf-local"
SHARED_HUB = "/content/drive/MyDrive/CDA-CACHE/huggingface/hub"
os.makedirs(LOCAL_HF, exist_ok=True)
os.environ['HF_HOME'] = LOCAL_HF

try:
  from google.colab import drive
  drive.mount('/content/drive')
except Exception as e:
  print("Google Drive not mounted.", e)

if os.path.isdir(SHARED_HUB):
  os.environ['HF_HUB_CACHE'] = SHARED_HUB
  print("Using the shared course cache for the models.")
else:
  print("The CDA-CACHE shortcut was not found in your Google Drive - see the setup instructions on Canvas.")
  print("The models will be downloaded for this session only.")

import transformers

try:
  import datasets
except:
  !pip install datasets
  import datasets

try:
  import accelerate
except:
  !pip install accelerate
  import accelerate

try:
  nltk.data.find('tokenizers/punkt_tab')
except LookupError:
  nltk.download('punkt')
  nltk.download('punkt_tab')

try:
  nltk.data.find('corpora/stopwords')
except LookupError:
  nltk.download('stopwords')


if not(os.path.exists('TwitterHate_Small.csv')):
  print('Downloading TwitterHate_Small.csv')
  !wget https://raw.githubusercontent.com/goto4711/cdai/main/TwitterHate_Small.csv



is_colab = "google.colab" in sys.modules
is_gpu_available = torch.cuda.is_available()

def display_library_version(library):
    print(f"Using {library.__name__} v{library.__version__}")

# Give visibility on versions of the core libraries
display_library_version(transformers)
display_library_version(datasets)
# Disable all info / warning messages
transformers.logging.set_verbosity_error()
datasets.logging.set_verbosity_error()


from transformers import set_seed
set_seed(42) # Set the seed to get reproducible results

if not is_gpu_available:
    print('------' * 10)
    print("No GPU was detected! This notebook can be *very* slow without a GPU 🐢")
    if is_colab:
        print("Go to Runtime > Change runtime type and select a GPU hardware accelerator.")


![Cultural Data Analysis · Week 4: Cultural Data Forms: Text](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week4-banner.png)

# Cultural Data Forms - Texts

*Workshop notebook* · Code-along: [goto4711.github.io/cda-teaching-tools/code-along/week4](https://goto4711.github.io/cda-teaching-tools/code-along/week4/) · Tools: [term frequency](https://goto4711.github.io/cda-teaching-tools/tools/text/), [from words to embeddings](https://goto4711.github.io/cda-teaching-tools/tools/embeddings/)

---

In the lecture, the dictionary method failed to detect hate speech convincingly. Today we try *Transformers* (https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)), the architecture behind ChatGPT and most of today's AI, introduced in the paper 'Attention is all you need' (https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf). Don't worry, it is not as complicated as it sounds.

We work with Hugging Face (https://huggingface.co/), the leading platform for open-source models and datasets. There is a lot of critical work on the concentration of power in AI, with a few companies owning the models and the cloud infrastructure they run on. Open-source models are one attempt to work against this.

> **Goals** · By the end of this workshop, you will be able to:
> 1. load a dataset of Tweets and turn its texts into tokens, the way language models read;
> 2. **fine-tune** a pretrained language model (DistilBERT) to detect hate speech, and compare it with the dictionary method;
> 3. understand the step from classification to **text prediction**, with a second model (T5) that answers in words;
> 4. do the steps of the lecture's traditional NLP again with language models, **without any training**.

> **To submit:** Run the whole notebook, complete every cell marked `#Complete cell`, and hand in the HTML file that the last cell makes. Sections and cells marked *optional* you can skip.

The figure below shows the Hugging Face pipeline, which we follow step by step. Once you know the steps, you can build many different applications with them.

<img alt="Hugging Face Pipeline" caption="A typical pipeline for training transformer models with the Datasets, Tokenizers and Transformers" src="https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg"/>

In the lecture, the dictionary method needed no training data. Now we use *supervised learning*: the model learns from Tweets that people have labelled as hate speech or not.

> **Background: Transformers and Hugging Face** (optional reading)
>
> With Transformers, machines can generate text, translate from one language to another, analyse comments, label the words in sentences, and much more. The paper that introduced them, 'Attention is all you need', has become one of the defining papers of the new AI age. We can do all the things from the lecture, from classification to information extraction, more easily and often much better with Transformers.
>
> Hugging Face makes working with Transformers quite easy. In many ways, its way of doing NLP is easier than what you saw in the lecture. Its ecosystem has three parts, which follow the figure above: *datasets* to load texts and prepare them for deep learning; *tokenizers*, much more efficient than the ones we have seen; and *models*. Most importantly, its models let us reuse the knowledge stored in other language models. This is called *fine-tuning*: we take an existing language model and adjust it to our task. We could, for instance, take the language model BERT, based on the original Transformers paper, and fine-tune it to recognise sentiments in the State of the Union speeches.

## Case Study: Hate speech on social media

### Loading the data

The Hugging Face hub has thousands of datasets (https://huggingface.co/datasets), including several on hate speech, such as `tdavidson/hate_speech_offensive`. `load_dataset("tdavidson/hate_speech_offensive")` would load it by its name. Most often, though, your data is not on the hub. We use a reduced and more balanced version of a Kaggle dataset of Tweets (https://www.kaggle.com/datasets/prakharprasad/twitter-hate-speech), stored as a CSV file. `label` is 1 for a Tweet containing hate speech and 0 otherwise; `tweet` contains the Tweet.

`load_dataset` loads a local file, too, with the `csv` script and `data_files="TwitterHate_Small.csv"`. `class_encode_column('label')` tells Hugging Face that 'label' is the target column. Run the next cell.

In [ ]:
#Keep cell
from datasets import load_dataset
ds = load_dataset("csv", data_files="TwitterHate_Small.csv")
ds = ds.class_encode_column('label')

The `load_dataset()` function is very powerful: https://huggingface.co/docs/datasets/en/loading. Let's take a look at the data next and print out `ds` and some data (the first 5 entries). Complete the cell below.

In [ ]:
#Complete cell
print(ds)
print(___['train'][:5])

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q1.hint()
#q1.solution()

As we are experts in machine learning by now, we know that we need a test dataset next to the training dataset. Actually, we would have also needed a validation dataset to make sure that our algorithm is good. It allows us to use the test data only for a final test and tune the model with the validation set. But we skip this here for learning reasons. You can read up on this at https://en.wikipedia.org/wiki/Training,_validation,_and_test_data_sets.

For the training and test data, we choose an 80-20 random split by running the cell below. Do you see how? Do you remember how we did this for sklearn? For the Hugging Face version with its `train_test_split()`, run the next cell.

In [ ]:
#Keep cell
hatespeech_local = ds['train']
hatespeech_local = hatespeech_local.train_test_split(test_size=0.2, seed = 42)
hatespeech_local

> **Background: datasets on the Hugging Face hub** (optional reading)
>
> The hub has many datasets on hate speech, because there is a lot of research on it. Try one at home:
> ```
> from datasets import load_dataset
> hatespeech_hf = load_dataset("tdavidson/hate_speech_offensive")
> train_ds = hatespeech_hf["train"]
> ```
> `hatespeech_hf` is a `DatasetDict`, which looks like a Python dictionary: the key `train` holds a training dataset of over 24,000 Tweets. A `Dataset` behaves like a Python list: `len(train_ds)` gives the number of rows, `train_ds[0]` the first example, `train_ds[:3]` the first three, and `train_ds["tweet"][:5]` the first five Tweets. `train_ds.column_names` lists the columns and `train_ds.features` their types.
>
> Its documentation (https://huggingface.co/datasets/tdavidson/hate_speech_offensive) shows who decided what is hate speech. Each Tweet was judged by at least three crowd workers: `hate_speech_count`, `offensive_language_count` and `neither_count` say how many of them chose each answer, and `class` is the majority vote: 0 for hate speech, 1 for offensive language and 2 for neither.
>
> Most often, your dataset is not on the hub. The hub version is actually a copy of a famous dataset on Kaggle that somebody imported. This is why we load a CSV file, which works the same way for your own data (https://huggingface.co/docs/datasets/en/process).

### From Datasets to DataFrames

Pandas is great for analysing data and visualising it. We can convert a `Dataset` to a Pandas `DataFrame` with the `set_format()` method. Run the cell below.

In [ ]:
#Keep cell
import pandas as pd
import numpy as np

hatespeech_local.set_format(type="pandas")
df_hs = hatespeech_local["train"][:]
df_hs.head()

Let's run a quick check whether everything is loaded.

In [ ]:
# Run this cell to test your code...
assert df_hs.shape[0] == 6548, f'Test failed'
print('All tests passed!')

As a reminder, label 0 stands for non-hateful and 1 for hateful. Let's add a column 'label_name' to reflect this. Do you remember `np.where`? Complete the cell below.

In [ ]:
#Complete cell
df_hs['label_name'] = np.where(df_hs['label'] == ___, 'Non-hateful', 'Hateful')
df_hs.head()

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q2.check()
#q2.hint()
#q2.solution()

> **Remember:** Whenever you are working with NLP classifications or classifications in general, it is good to start with understanding the distribution across the target classes. We cannot cover this here, but skewed classes have to be addressed in very different ways from more even distributions.

Let's visualise `label_name`, which is our target variable. Complete the cell below.

In [ ]:
#Complete cell

import matplotlib.pyplot as ___

df_hs["label_name"].value_counts(ascending=True).plot.barh()
plt.title("Frequency of Classes")
plt.___()

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q3.hint()
#q3.solution()

The dataset is imbalanced: there are far fewer hateful Tweets. Keep this in mind for the baseline below.

Tweets are short, around 15 words. That matters for language models: they have a *maximum context size*, the number of tokens they can see at once. For DistilBERT, the model we use (https://huggingface.co/docs/transformers/en/model_doc/distilbert), it is 512 tokens, so our Tweets fit easily. Longer texts would have to be cut or split into chunks.

We don't need the dataframe format any more, so we reset it: run `hatespeech_local.reset_format()`.

---

## From Text to Tokens

Language models cannot process strings. They need texts broken into *tokens* and turned into numbers. In the lecture, you split texts into words. Language models use a cleverer split, and they *learn* it from the texts they were trained on. Let's compare the options.

### Character Tokenization

The simplest option is to treat each character as a token. In Python, a string is a collection of characters, so this takes one line. Run the cell below.

In [ ]:
#Keep cell
text = "Tokenizing text is fun but also strange. Even poetry becomes a number!"
tokenized_text = list(text)
print(tokenized_text)

Afterwards, we need to convert each character into an integer, a process  called _numericalization_. Deep learning/transformers requires all input to be numbers. Numericalisation is easy for characters by simply assigning them a unique integer. The code below is typical for Transformers. It creates a dictionary, which maps tokens (here characters) to numbers by assigning an integer to them - in alphabetical order of the tokens (`sorted`). Run the cell.

In [ ]:
#Keep cell
token2idx = {ch: idx for idx, ch in enumerate(sorted(set(tokenized_text)))}
print(token2idx)

We can now use `token2idx` to transform the tokenized text to a list of integers. Complete the cell below.

In [ ]:
#Complete cell
input_ids = [token2idx[token] for token in ___]
print(input_ids)

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q5.check()
#q5.hint()
#q5.solution()

### Word Tokenization

Instead of characters, we can split the text into words, as in the lecture, and give each word a number. Run the cell below.

In [ ]:
#Keep cell
tokenized_text = text.split()
print(tokenized_text)

> **Background: the problems of word tokens** (optional reading)
>
> Splitting on whitespace, as `split()` does, is the simplest word tokenizer. It has several problems. It does not consider the position of a word and its context. And the vocabulary becomes very large very quickly, which requires very big neural networks that are expensive to train.
>
> A common workaround is to limit the vocabulary to, say, the 100,000 most common words. All other words become 'unknown' and are mapped to a shared `UNK` token. You will meet this approach quite often, but we lose potentially important information this way.
>
> Subword tokenization combines the best of characters and words: rare words are split into smaller units, so that the model can deal with complex words and misspellings, while frequent words stay whole, which keeps the inputs short. Like many modern word tokenizers, it is *learned* from the training texts, with a mix of statistical rules and algorithms, a little like Word2Vec in the lecture. Subword tokenization is easy with Hugging Face.

With words, the vocabulary quickly becomes very large, and every word the model has not seen becomes 'unknown'. With characters, texts become very long and the single tokens mean little. *Subword tokenization* is the compromise: frequent words stay whole, rare words are split into smaller pieces. The pieces are learned from the training texts.

### Subword Tokenization

BERT and DistilBERT use the subword tokenizer WordPiece (https://ieeexplore.ieee.org/document/6289079). `AutoTokenizer` loads the tokenizer that belongs to a model, from its *checkpoint*, the saved version of the model. We use `distilbert-base-cased`, which keeps the difference between upper and lower case (https://huggingface.co/distilbert/distilbert-base-cased). Run the cell below.

In [ ]:
#Keep cell
from transformers import AutoTokenizer

checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

> **Background: auto classes and checkpoints** (optional reading)
>
> `AutoTokenizer` belongs to a larger set of 'auto' classes (https://huggingface.co/docs/transformers/model_doc/auto), whose job is to retrieve a model's configuration, weights, etc. automatically from its checkpoint. This makes it easy to switch between models. A *checkpoint* is a saved version of a model; think of it as the latest save. At the beginning, rely on the auto classes as much as possible. You could also load DistilBERT's tokenizer directly:
> ```
> from transformers import DistilBertTokenizer
> distilbert_tokenizer = DistilBertTokenizer.from_pretrained(checkpoint)
> ```

Now let's see `tokenizer` in action with our `text`. Complete the cell below.

In [ ]:
#Complete cell
encoded_text = ___(text)
print(encoded_text)

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q6.hint()
#q6.solution()

Every subword has now a unique integer in the `input_ids` field. You can ignore `attention_mask`. With `input_ids`, we can convert them back into tokens by using the tokenizer's `convert_ids_to_tokens()` method. Run the next cell.

In [ ]:
#Keep cell
tokens = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)
print(tokens)

Let's run a quick test that all is good.

In [ ]:
# Run this cell to test your code...
assert tokens[-1]=='[SEP]', f'Test failed'
print('All tests passed!')

From the output, we can observe three things:
1. Some special tokens, `[CLS]` and `[SEP]`, have been added to the start and end of the sequence. These tokens differ from model to model, but their main role is to indicate the start and end of a sequence.
1. The tokens have kept their upper and lower case, which is a feature of this cased version of DistilBERT.
1. Finally, we can see that the word 'Tokenizing', e.g., has been split into three tokens (`To`, `##ken`, `##izing`), since it is not a common word. The `##` prefix in `##izing` means that the preceding string is not a whitespace. With the prefix, we can convert them back.

The `AutoTokenizer` class has a `convert_tokens_to_string()` method for converting tokens back. Complete the next cell.

In [ ]:
#Complete cell
tokenizer.convert_tokens_to_string(___)

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q7.hint()
#q7.solution()

DistilBERT's vocabulary has about 29,000 tokens (`tokenizer.vocab_size`), and it can read up to 512 tokens at once (`tokenizer.model_max_length`).

### Tokenizing the Whole Dataset

To tokenize the whole dataset, we use the `map()` method of our `DatasetDict` with a function that tokenizes a batch of examples. Run the cell below to define it.

In [ ]:
#Keep cell
def tokenize(batch):
    return tokenizer(batch["tweet"], padding=True, truncation=True)

`padding=True` fills shorter Tweets with zeros, the `[PAD]` token, up to the length of the longest one in a batch, and `truncation=True` cuts texts that are longer than the model's maximum context size. Run the cell below to see `tokenize()` on two Tweets; you can ignore `attention_mask`.

In [ ]:
#Keep cell
tokenize(hatespeech_local["train"][:2])

> **Background: padding and special tokens** (optional reading)
>
> In the output above, the second Tweet is shorter than the first, so zeros have been added to 'pad' it to the same length. The zeros stand for the special token `[PAD]`. Other special tokens are `[CLS]` and `[SEP]`, which mark the start and end of a sequence, and `[UNK]` for unknown tokens. `tokenizer.all_special_tokens` and `tokenizer.all_special_ids` list them.
>
> Always set `padding=True` and `truncation=True` when you write your own `tokenize` function. To tokenize another dataset, you only need to change the name of the text column in it.

With the tokenization function `tokenize(batch)`, we can apply it across all the splits in the dataset very easily with `map()`. We print the column_names of 'test' to demonstrate our success. Run:
```
hatespeech_encoded = hatespeech_local.map(tokenize, batched=True, batch_size=None)
print(hatespeech_encoded["test"].column_names)
```

Hugging Face's Transformers assume that your target variable has the column name 'labels' and the textual input is 'text'. Therefore, we need to rename the columns with `rename_columns()`. Run the cell below.

In [ ]:
#Keep cell
hatespeech_encoded = hatespeech_encoded.rename_columns({'label':'labels', 'tweet': 'text'})
print(hatespeech_encoded["test"].column_names)

To make training faster in class, we keep only half of the training and test data. Run the cell below.

In [ ]:
#Keep cell
# To make training faster in class, we keep a random half of the training and the test data
hatespeech_shuffled = hatespeech_encoded.shuffle(seed=42)   # shuffles every split at once
hatespeech_encoded_train_small = hatespeech_shuffled["train"].take(hatespeech_shuffled["train"].num_rows // 2)
hatespeech_encoded_test_small = hatespeech_shuffled["test"].take(hatespeech_shuffled["test"].num_rows // 2)

Let's print out `hatespeech_encoded_train_small` and `hatespeech_encoded_test_small` to check that everything looks alright.

---

## Transformer Models

DistilBERT was pretrained by predicting hidden words: in 'The ___ has a garden.', it learns that 'house' is likely. This teaches it a lot about language in general, but nothing about hate speech. So we *fine-tune* it: we train it a little more, on our labelled Tweets.

### Baseline

First, a baseline to compare with: the dictionary method from the lecture, on our Tweets. Run the next cell; you know the code from the lecture.

In [ ]:
# Keep cell
# Repeat dictionary-based sentiment analysis

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stopwords_ = list(stopwords.words('english'))
neg_words = list(set(list(pd.read_csv('https://raw.githubusercontent.com/goto4711/social-cultural-analytics/master/hate-vocab-eng.csv')['word'])))

def hate_score_(text):
    score = 0
    words = [word.lower() for word in word_tokenize(text)]
    for word in words:
        if word in neg_words:
            score += 1
    return score

df_hs = df_hs.sample(frac=0.5, random_state=1)   # half of the tweets, as for the model above

df_hs['hate_score'] = df_hs['tweet'].apply(hate_score_)
df_hs['hate_predict'] = np.where((df_hs['hate_score'] > 0), 'Hateful', 'Non-hateful')

df_hs["hate_predict"].value_counts(ascending=True).plot.barh()
plt.title("Frequency of Classes")
plt.show()

`hate_score_` counts the number of times a word from the hatespeech dictionary appears in the Tweets. It assigns that count to `df_hs['hate_score']`. If the hate_score is larger than 0, the Tweet is predicted to be 'Hateful' and otherwise 'Non-hateful'.

Let's calculate the accuracy:

In [ ]:
#Keep cell
from sklearn.metrics import accuracy_score

accuracy_score(df_hs["label_name"], df_hs["hate_predict"])

The accuracy of about 0.73 looks ok, but it is no better than always guessing the dominant 'Non-hateful' class, which would give about 0.73 as well. It comes almost entirely from predicting 'Non-hateful' correctly, as the confusion matrix shows. Run the cell below.

In [ ]:
#Keep cell
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(df_hs["label_name"], df_hs["hate_predict"], labels=["Non-hateful", "Hateful"])
pd.DataFrame(cm, columns = ["Non-hateful", "Hateful"], index = ["Non-hateful", "Hateful"])

> **Remember:** The dictionary hardly finds any hateful Tweets and is not useful in practice. Be very sceptical when companies present only accuracy to praise their models.

> **Background: GPUs** (optional reading)
>
> Graphics processing units are the basic hardware of language modelling: they calculate billions of operations efficiently and in parallel. 'cuda' is the GPU on Colab, 'mps' the one on recent Macs (https://raoofnaushad7.medium.com/difference-between-cuda-and-mps-3d2596629c89).

Language models run much faster on a GPU (a graphics card). The cell below picks the GPU if there is one ('cuda' on Colab, 'mps' on a Mac) and the CPU otherwise. Run it.

In [ ]:
#Keep cell

if torch.cuda.is_available():
    DEFAULT_DEVICE = "cuda"
else:
    if torch.backends.mps.is_available():
        DEFAULT_DEVICE = "mps"
    else:
        DEFAULT_DEVICE = "cpu"
device = torch.device(DEFAULT_DEVICE)
print(DEFAULT_DEVICE)

> **Important:** It should print out the device you are using. In my case it would be 'mps' for my home computer and 'cuda' on Google Colab if you connected the runtime correctly. If you see 'cpu', start from the beginning, connect the right runtime and run all cells above again.

Let's go on and finally fine-tune.

### Fine-Tuning Transformers

We want to fine-tune the pretrained DistilBERT model and first need to get it from `AutoModelForSequenceClassification`. The Hugging Face `AutoModelForSequenceClassification` model has a so-called classification _head_ on top of the pretrained model body, which can be trained. We just need to specify how many labels the model has to predict since this dictates the number of outputs the classification head has. We set `num_labels=2` for the binary choice of hate_speech or not.

Run the cell below. The most important line is `model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2).to(device)`, which uses the `checkpoint = "distilbert-base-cased"` from the tokenizer above to load `AutoModelForSequenceClassification` for a binary classification (`num_labels=2`). `to(device)` sends the calculations to our default device, which should be cuda on Colab.

In [ ]:
#Keep cell
from transformers import AutoModelForSequenceClassification   # DistilBERT with a classification head on top
# Load the pretrained model; num_labels=2 gives the head two outputs: non-hateful and hateful
# .to(device) moves the model to the GPU (cuda) if there is one
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2).to(device)

Next, we define how to measure the model during training: `compute_metrics()` computes the accuracy, so that we can compare with the baseline. Run the cell below. You may see warnings about Hugging Face versions; ignore them.

In [ ]:
#Keep cell
from sklearn.metrics import accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids                  # the true labels of the test tweets
    preds = pred.predictions.argmax(-1)      # for each tweet, the class with the higher score
    acc = accuracy_score(labels, preds)      # the share of tweets the model got right
    return {"accuracy": acc}                 # the Trainer expects a dictionary: name -> value

> **Background: metrics and training settings** (optional reading)
>
> **Metrics.** Hugging Face's `Trainer` calls `compute_metrics()` to evaluate the model during fine-tuning. It receives `pred`, with the `predictions` (one score per class for each Tweet) and the true `label_ids`, and returns a dictionary that maps the name of each metric to its value. You can reuse this function for most Hugging Face classifications.
>
> **Training settings.** `TrainingArguments` gives fine-grained control over training. Most of its arguments are for reporting, and you can copy them next time. `output_dir` is where all the files from training are stored. `batch_size` has a lot to do with what your machine can hold in memory at once: at the beginning, don't go beyond 128. If you see strange memory errors, first reduce the batch size to 64 or 32; training then takes longer, as fewer Tweets are processed in each pass. The more *epochs*, the more chances the network has to learn, but the longer it takes; we use 2 for demonstration. `learning_rate=2e-5` sets how fast the model's parameters change.

### Training the model

`TrainingArguments` holds the settings for training (https://huggingface.co/docs/transformers/main_classes/trainer). Three matter most: `batch_size`, how many Tweets the model sees before it updates itself (if you get memory errors, make it smaller: 64 or 32); `epochs`, how often it goes through all the training Tweets; and `learning_rate`, how big each update is. Read the comments in the cell below: they explain each line. Then run it.

In [ ]:
#Keep cell
from transformers import TrainingArguments

batch_size = 128     # how many tweets the model looks at before it updates itself
epochs = 2           # how many times it goes through all the training tweets

training_args = TrainingArguments(output_dir=f"{checkpoint}-finetuned-emotion",   # where the results are saved
                                  num_train_epochs=epochs,
                                  learning_rate=2e-5,                      # how big each update step is
                                  per_device_train_batch_size=batch_size,
                                  per_device_eval_batch_size=batch_size,
                                  weight_decay=0.01,                       # keeps the model from over-fitting
                                  eval_strategy="epoch",                   # test after every epoch
                                  disable_tqdm=False,                      # show a progress bar
                                  report_to="none")                        # don't send logs anywhere


Now we fine-tune the model with the `Trainer`. Run the next cell. This takes a while: you are now doing really advanced stuff.

In [ ]:
#Keep cell
from transformers import Trainer

trainer = Trainer(model=model,                                   # the model to fine-tune
                  args=training_args,                            # the settings from the cell above
                  compute_metrics=compute_metrics,               # how to measure it: accuracy
                  train_dataset=hatespeech_encoded_train_small,  # the tweets it learns from
                  eval_dataset=hatespeech_encoded_test_small,    # the tweets it is tested on
                  processing_class=tokenizer)                    # the tokenizer that belongs to the model
trainer.train();      # start training: this is where the model learns

The logs show an accuracy on the test data of more than 88%, a great improvement on the dictionary. To see where the model goes right and wrong, we need its predictions for the test Tweets. Run the cell below.

In [ ]:
#Keep cell
preds_output = trainer.predict(hatespeech_encoded_test_small)

Run another test to check that you are on the right track.

In [ ]:
# Run this cell to test your code...
assert len(preds_output) == 3, f'Test failed'
print('All tests passed!')

For each Tweet, the model gives two scores, one for 'Non-hateful' and one for 'Hateful', and we pick the larger one. `np.argmax()` returns the position of the larger score: 0 or 1. Type in `y_preds = np.argmax(preds_output.predictions, axis=1)` to do this for all Tweets at once (`axis=1` compares the scores within each row), and print the first ten with `y_preds[:10]`.

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q8.check()
#q8.hint()
#q8.solution()

Run another test.

In [ ]:
# Run this cell to test your code...
assert len(y_preds) > 810, f'Test failed'
print('All tests passed! Part 1 complete!')

With the predictions, we can plot the confusion matrix. Complete the cell below by looking back at the earlier example or using the documentation for `confusion_matrix`.

In [ ]:
#Complete cell
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_test = np.array(hatespeech_encoded_test_small["labels"])

cm = ___(y_test, y_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=___)
disp.plot()
plt.show()

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q9.hint()
#q9.solution()

This is much better: the model can actually find hateful Tweets (label 1).

To try the model on Tweets of your own, a Hugging Face `pipeline` (https://huggingface.co/docs/transformers/en/main_classes/pipelines) puts the tokenizer and the model together. Run the next two cells.

In [ ]:
#Keep cell
from transformers import TextClassificationPipeline
# A pipeline puts tokenizer and model together: text in, label out
# model.to('cpu') moves the model back to the CPU, where the tokenizer is
pipe = TextClassificationPipeline(model=model.to('cpu'), tokenizer=tokenizer)

In [ ]:
#Keep cell
nice_tweet = "'i am looking forward to seeing it!! @user @user'"
print('Nice Tweet: ', pipe(nice_tweet, top_k=None))

bad_tweet = "black genius v white supremacy are you missing the action? the #freemasons being taught a lesson about .'."
print('Bad Tweet: ', pipe(bad_tweet, top_k=None))

> **Background: pipelines and inference** (optional reading)
>
> Using a trained model on new texts is called *inference*. A Hugging Face `pipeline` (https://huggingface.co/docs/transformers/en/main_classes/pipelines) brings several steps together: here, it 'pipes' the tokenizer into the model. The model has to be moved back to the CPU first (`model.to('cpu')`), because the tokenizer and the model must be on the same device.
>
> Our steps follow the first chapter of the Transformers book by the Hugging Face team (Tunstall, von Werra and Wolf, 2022). We changed the datasets and the task and removed some content. The book is a great introduction, but changes to the Hugging Face libraries quickly make parts of it outdated.
>
> Does all this look scary? It is always the same steps. Gemini can write them too: open the Gemini chat in Colab and ask it to "classify from ds the label from tweet using Hugging Face's DistilBERT". Compare its steps with ours.

The model predicts non-hateful content for `nice_tweet` ('LABEL_0' has the higher score) and hate speech for `bad_tweet`. Try some other Tweets!

> **Remember:** You have now fine-tuned a Transformer to classify Tweets. The steps from the figure at the top are always the same: load the data, tokenize it, load a pretrained model, train, evaluate. They follow the first chapter of _Lewis Tunstall, Leandro von Werra, Thomas Wolf (2022): Natural Language Processing with Transformers, O'Reilly Media, Inc._, a great introduction by the Hugging Face team.

---

## A Second Case Study: Do Two Sentences Mean the Same Thing --- One Recipe for Every Task

Here is a twist that shows the real power of language models. Until now, different NLP tasks needed different kinds of model:

* classification tasks (like sentiment analysis) used models that output a class label — a number;
* translation models output target-language sentences;

A different architecture was needed for each job. The famous *T5 paper* ([Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://arxiv.org/pdf/1910.10683)) showed you can collapse all of these into one idea: *text in → text out*. The trick is to turn the labels into text too. Then a single model, with a single training procedure and a single piece of code, can handle classification, translation, summarisation — everything.

Why would we want this? Because instead of maintaining many specialised models, we get one general recipe. The task simply becomes "phrase your problem as text-to-text."

Compare the two case studies in this notebook:

| | Case 1: DistilBERT (hate speech) | Case 2: T5 (paraphrases) |
|---|---|---|
| What goes in | a tweet | the task and two sentences, written as one text: `mrpc sentence1: ... sentence2: ...` |
| What comes out | one score per class | a word, written by the model |
| The label | a number: `1` | a word: `equivalent` |
| To do a new task | add a new head and train it | write the new task into the input text |

This is the step from classification to text prediction, and it is the idea behind the language models in the rest of this notebook and in the group exercise.

At the end of this section, you will be able to ask the model about two sentences of your own, and it will answer in words:

```
Model says: 'equivalent'
   A: The committee approved the new budget.
   B: The new budget was approved by the committee.
```

We will show it with a classification task — deciding whether two sentences are paraphrases — but instead of a model that outputs `0` or `1`, we'll fine-tune T5 to literally write the word `equivalent` or `non_equivalent`. This is one of the exact tasks the T5 paper used. So, here you also learn something about reproducing papers.

The dataset is *MRPC* (the Microsoft Research Paraphrase Corpus) of pairs of sentences labelled as equivalent in meaning, or not: https://www.kaggle.com/datasets/doctri/microsoft-research-paraphrase-corpus.

> **Read more:** Side note: I also chose this dataset because I used it to understand benchmarking in the new economy of enrichment of GenAI: https://journals.sagepub.com/doi/10.1177/20539517261451463. The actual experiments with MRPC did not make it into the paper because of the word-count limit but they taught me a lot about how AI benchmarking really works.

To keep the focus on the *transfer*, we repeat the same steps as before (model, tokenizer, `TrainingArguments`, `Trainer`) and reuse the `device`; only the checkpoint changes to T5 (`t5_checkpoint`, `t5_tok`, `t5_args`).

Run the next cell to set up the data.

In [ ]:
#Keep cell
from datasets import load_dataset

# "nyu-mll/glue" is the current home of the classic GLUE datasets.
mrpc = load_dataset("nyu-mll/glue", "mrpc")
mrpc

`mrpc` is a `DatasetDict` with three splits. The `test` split keeps its labels hidden (it was used for a public leaderboard), so we will train on `train` and use the `validation` split as our held-out test set. Let's look at one example by running the next cell.

In [ ]:
#Keep cell
mrpc["train"][0]

So each row gives us `sentence1`, `sentence2`, and a `label`. Try reading a few by changing the index number. You can decide for yourself whether they say the same thing, which is exactly what we will teach the model to do.

Now we turn both the input and the label into text. This is the heart of the T5 idea. We build an input string from the two sentences, and we turn the numeric label into a word: `equivalent` for `1`, `non_equivalent` for `0`. The model will be trained to read the input text and generate that label word.

The comprehensions and `zip` are given. Fill in the four gaps: the two sentences `s1` and `s2` in the input text; the label word for `0`, which is where the label stops being a number; and the method that applies `make_text` to every row of the dataset (you used it to tokenize the whole hate-speech dataset).

Complete the next cell.

In [ ]:
#Complete cell
def make_text(batch):
    return {
        # the input: the task name and both sentences, written as one text
        "input_text": ["mrpc sentence1: " + ___ + " sentence2: " + ___
                       for s1, s2 in zip(batch["sentence1"], batch["sentence2"])],
        # the label: 1 becomes the word "equivalent", 0 becomes ...
        "target_text": ["equivalent" if y == 1 else ___
                        for y in batch["label"]],
    }

mrpc = mrpc.___(make_text, batched=True)   # apply make_text to every row of the dataset

print("INPUT :", mrpc["train"][0]["input_text"])
print("LABEL :", mrpc["train"][0]["target_text"], ".  This label is now text, not a number")

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q10.check()
#q10.hint()
#q10.solution()

The second part of this exercise is to show that we always repeat the same steps if we fine-tune a model with Hugging Face.

**Step 1 — Model**

In the hate-speech exercise we used `AutoModelForSequenceClassification` (a classification that outputs class numbers). Here, we use `AutoModelForSeq2SeqLM` — a text-to-text model with T5. That one change is the whole point. The model now reads text and generates text.

Complete the next cell.

In [ ]:
#Complete cell
from transformers import AutoTokenizer, ___

t5_checkpoint = "google-t5/t5-small"
t5_tok = AutoTokenizer.from_pretrained(t5_checkpoint)
model_t5 = ___.from_pretrained(t5_checkpoint).to(device)

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q11.check()
#q11.hint()
#q11.solution()

**Optional, before training.** The original T5 was trained on a mix of tasks, and this paraphrase task was one of them. Ask the model before we train it. Whatever it answers, notice that the answer is text, not a number.

In [ ]:
#Keep cell
# One pair of sentences, written in the task format
inp = "mrpc sentence1: The committee approved the new budget. sentence2: The new budget was approved by the committee."
ids = t5_tok(inp, return_tensors="pt").input_ids.to(device)   # text -> token ids
out = model_t5.generate(ids, max_new_tokens=5)                # the model writes up to 5 tokens
print(t5_tok.decode(out[0], skip_special_tokens=True))        # token ids -> text

**Step 2 — Tokenize**

Same step as always, but now we tokenize both the input text and the target text (the label word). `text_target=` tells the tokenizer it is encoding the thing the model should learn to write.

Run the cell.

In [ ]:
#Keep cell
def encode(batch):
    # tokenize the input text
    enc = t5_tok(batch["input_text"], truncation=True, max_length=128)
    # tokenize the label word too: this is what the model should learn to write
    enc["labels"] = t5_tok(text_target=batch["target_text"], truncation=True, max_length=8)["input_ids"]
    return enc

# apply encode to every row, and keep only the token columns
mrpc_enc = mrpc.map(encode, batched=True, remove_columns=mrpc["train"].column_names)
mrpc_enc["train"][0].keys()

A quick check that nothing went wrong.

In [ ]:
# Run this cell to test your code...
assert mrpc_enc["train"].num_rows == 3668, 'Test failed'
print('All tests passed!')

**Step 3 — Train**

Exactly the familiar `Trainer` workflow. The only new ingredient is `DataCollatorForSeq2Seq`, which knows how to pad text labels. T5 likes a slightly higher learning rate than the classifier did, so we set one here.

Run the next cell.

In [ ]:
#Keep cell
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

t5_args = TrainingArguments(
    output_dir="t5-mrpc",              # where the results are saved
    num_train_epochs=3,                # three passes through the training pairs
    learning_rate=3e-4,                # T5 likes bigger steps than DistilBERT
    per_device_train_batch_size=16,
    weight_decay=0.01,
    report_to="none",
)

# The collator pads the texts in a batch to the same length; the only new ingredient
collator = DataCollatorForSeq2Seq(t5_tok, model=model_t5)

trainer_t5 = Trainer(
    model_t5,                          # the model, as before
    t5_args,                           # the settings, as before
    train_dataset=mrpc_enc["train"],   # the sentence pairs it learns from
    data_collator=collator,
    processing_class=t5_tok,
)
trainer_t5.train()

**Step 4 — Evaluate**

Now we evaluate with the held-out validation set.

Run the next cell.

In [ ]:
#Keep cell
import numpy as np

val = mrpc["validation"]   # pairs the model has not seen
# the right answers, as words
gold = ["equivalent" if y == 1 else "non_equivalent" for y in val["label"]]

preds = []
for i in range(0, len(val), 32):                       # 32 pairs at a time
    enc = t5_tok(val["input_text"][i:i+32], return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    out = model_t5.generate(**enc, max_new_tokens=5)   # the model writes its answers
    preds.extend(t5_tok.batch_decode(out, skip_special_tokens=True))   # token ids -> words

# accuracy: the share of answers that are exactly the right word
acc = np.mean([p.strip() == g for p, g in zip(preds, gold)])
print(f"Held-out accuracy: {acc:.3f}")
print("Raw text the model generated (first 5):", [p.strip() for p in preds[:5]])

The model is producing the words `equivalent` or `non_equivalent`, not class numbers. We score it by comparing those words to the true label words.

Try it on your own sentences: change the sentences in `pairs` in the next cell. You can observe the model returns a word, not a number. That is the text-to-text idea made concrete.

Run the next cell.

In [ ]:
#Keep cell
def classify_pair(s1, s2):
    inp = "mrpc sentence1: " + s1 + " sentence2: " + s2                # the task, written as text
    ids = t5_tok(inp, return_tensors="pt", truncation=True, max_length=128).input_ids.to(device)
    out = model_t5.generate(ids, max_new_tokens=5)                     # the model writes its answer
    return t5_tok.decode(out[0], skip_special_tokens=True)             # token ids -> the answer word

pairs = [
    ("The committee approved the new budget.", "The new budget was approved by the committee."),
    ("She is reading a book in the garden.",   "The train leaves Berlin at noon."),
]
for a, b in pairs:
    print(f"Model says: {classify_pair(a, b)!r}")
    print(f"   A: {a}")
    print(f"   B: {b}\n")

**Think about it:** the model you have just trained answers in words. What would you need to change so that the same model and the same code translate English into German instead of deciding whether two sentences mean the same thing?

Run the next cell for a hint. To see the answer, remove the `#` in front of `q12.solution()` and run the cell again.

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q12.hint()
#q12.solution()

> **The answer, in short:** Only the data changes. The input text names the new task, e.g. `"translate English to German: That is good."`, and the target text is the German sentence, `"Das ist gut."`. The model (`AutoModelForSeq2SeqLM`), the tokenizer, `TrainingArguments`, the `Trainer` and `generate()` all stay the same. The original T5 was trained with exactly this prefix, among many other tasks.

> **Remember:** We just solved a classification problem with a text generation model, by writing the labels as text. That unification, set out in the T5 paper, is a large part of why today's general-purpose language models are so powerful.

Keep this idea in mind for the rest of the notebook and for the group exercise. The chat models below, and the large language models you query through their APIs, work on the same principle at a much larger scale: the task is written into the input text, and the answer comes back as text.

---

## The lecture again, with language models

In the lecture, the computer read texts step by step: tokens, cleaning, counting, entities, dictionaries, topics, word vectors. Language models can do all of these tasks too, mostly without any training: we give them only the task. This is called *zero-shot* learning (https://en.wikipedia.org/wiki/Zero-shot_learning).

| Lecture: traditional NLP | Workshop: language models |
|---|---|
| 1. Tokens: `word_tokenize` | Subword tokens, learned from texts (the first part of this notebook) |
| 2. Cleaning: stopwords remove "not" | No cleaning: the model reads every token, including "not" |
| 3. Counting and tf-idf: distinctive words | A prompt: "list the five most important words" |
| 4. Entities with spaCy | An entity model, or a prompt: "list every country" |
| 5a. A dictionary of hate words | A sentiment model; our fine-tuned DistilBERT; a chat prompt |
| 5b. Topic models | Zero-shot classification with your own labels, or a prompt: "what are the main topics?" |
| 5c. Word2Vec neighbours | A prompt: "which words are closest in meaning to 'freedom'?" |
| Not in the lecture | Translating, summarising, answering questions |

We use two texts: the lecture's example sentence from Obama, and the opening of the 2025 State of the Union Address by Ursula von der Leyen, President of the European Commission. Run the cell below.

In [ ]:
#Keep cell
# The lecture's example sentence, and the opening of von der Leyen's 2025 address
example = "And that is why I can stand here tonight and say without exception or equivocation that the United States of America does not torture."

text_sotu_europe = """Honourable Members,
Europe is in a fight. A fight for a continent that is whole and at peace. For a free and independent Europe. A fight for our values and our democracies. A fight for our liberty and our ability to determine our destiny for ourselves. Make no mistake – this is a fight for our future.
I thought long and hard about whether to start this State of the Union address with such a stark appraisal. After all, we Europeans are not used to – or comfortable with – talking in such terms. Because our Union is fundamentally a peace project. But the truth is that the world of today is unforgiving. And we cannot varnish over the difficulties that Europeans feel every day. They can feel the ground shift beneath them. They can feel things getting harder just as they are working harder. They can feel the impact of the global crisis. Of the higher cost of living. They feel the speed of change affecting their lives and careers. And they worry about the endless spiral of events they see on the news – from the devastating scenes in Gaza to the relentless Russian barrage on Ukraine.
"""

### Cleaning and sentiment: steps 2 and 5a

In the lecture, cleaning turned Obama's sentence into "united states america torture", and a dictionary counted words from a list. The cell below shows how DistilBERT's tokenizer reads the sentence, and what a sentiment model, trained on film reviews, makes of the sentence and of von der Leyen's opening. Run it.

In [ ]:
#Keep cell
from transformers import pipeline
pipe_device = 0 if torch.cuda.is_available() else -1   # run the pipelines on the GPU if there is one

# No cleaning: the tokenizer keeps every word, including "not"
print(tokenizer.tokenize(example))

# A ready-made sentiment model: positive or negative, with a score
sentiment = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", revision="714eb0f", device=pipe_device)
print(sentiment(example))
print(sentiment(text_sotu_europe))

Is a statement against torture positive or negative? The model has to choose one of its two labels. Remember that our fine-tuned DistilBERT above also did step 5a, for hate speech.

> **Remember:** In this part, `pipeline()` gets a few extra arguments: `model=` and `revision=` name exactly the model version stored in our shared course cache, and `device=pipe_device` runs it on the GPU. In your own work you can leave them out, e.g. `pipeline("text-classification")`, and Hugging Face picks a suitable default model.

### Topics: step 5b

Instead of topics learned from word counts, a zero-shot classifier decides how well each of *your own* labels fits a text. Complete the cell below with a list of labels (strings) that you want to test.

In [ ]:
#Complete cell
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", revision="d7645e1", device=pipe_device)

# Choose your own categories to sort the speech into (at least two):
candidate_labels = [___]

classifier(text_sotu_europe, candidate_labels=candidate_labels)

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q13.check()
#q13.hint()
#q13.solution()

The results depend a lot on your labels. I tried `["politics", "economy", "war", "climate", "technology"]`: politics and war stand out, and technology clearly plays no role in this opening (its score is close to 0). Unlike a topic model, you decide the topics, and the model only weighs them.

### Entities: step 4

The cell below runs a model trained to find people, places and organisations, like spaCy in the lecture. Run it.

In [ ]:
#Keep cell

ner_tagger = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", revision="4c53496", aggregation_strategy="simple", device=pipe_device)
outputs = ner_tagger(text_sotu_europe)
pd.DataFrame(outputs)

### A chat model

Now a small chat model, Qwen, open-sourced by Alibaba. Text generation models continue a text. Run the cell below: it loads the model and writes a reply to the speech.

> **Hint:** This is a very small model, so the results are not great. With time, try a bigger one: `model_name = 'Qwen/Qwen2.5-1.5B-Instruct'`.

In [ ]:
#Keep cell

model_name = 'Qwen/Qwen1.5-0.5B-Chat'

generator = pipeline("text-generation", model = model_name, device=pipe_device)
response = "Dear speaker, it is great to hear your vision for Europe."
prompt = text_sotu_europe + "\n\nResponse:\n" + response
outputs = generator(prompt, max_length=400)
print(outputs[0]['generated_text'])

Chat models are prompted with `messages`, the standard format of chatbots: a *system* message with the instruction, and a *user* message with the text. Do you recognise it from ChatGPT or Gemini? Run the next cell to classify the sentiment of the speech (https://huggingface.co/docs/transformers/en/tasks/prompting).

In [ ]:
#Keep cell
# The task as a chat: the instruction as a system message, the speech as a user message
messages = [
    {"role": "system", "content": "You are a helpful assistant. Classify the sentiment of the provided text. Respond with only one word: 'positive', 'negative', or 'neutral'."},
    {"role": "user", "content": f"Text: \"{text_sotu_europe}\"\nSentiment:"}
]
# The pipeline turns the messages into the model's own chat format and generates a reply
output = generator(messages, max_new_tokens=5, do_sample=False)
# The reply is the last message in the conversation
print(output[0]["generated_text"][-1]["content"])

**Optional, what the model really receives.** The model does not see the list of messages. A chat template turns it into one long text with special markers such as `<|im_start|>`. Run the cell below to see it.

In [ ]:
#Keep cell
tokenizer = AutoTokenizer.from_pretrained(model_name)   # the tokenizer of the Qwen model
# apply_chat_template writes the messages as the single text the model is trained on
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Everything is now a matter of the instruction. **Redo one step from the lecture with a prompt.** Change the content of the message in the cell below so that the model does one of these tasks with the speech:
- step 3: list the five most important words of the speech;
- step 4: list every country mentioned;
- step 5b: name the three main topics;
- step 5c: list the words closest in meaning to 'freedom' in political speeches;
- or something the lecture could not do: translate the speech into German, summarise it in one sentence, or answer a question about it ("Using only this text, answer the question: ...").

Complete the cell below.

In [ ]:
#Complete cell
messages = [
    {"role": "user",
     "content": f"___\n\n{text_sotu_europe}"}
]
output = generator(messages, max_new_tokens=80)
print(output[0]["generated_text"][-1]["content"])

In [ ]:
#Keep cell. To see a hint or the solution, remove the # in front of that line.
q14.check()
#q14.hint()
#q14.solution()

> **What can we no longer check?** Compare the answers with the lecture's tools. Asked for every country, our small model listed more than a dozen European countries, from Norway to Portugal, that the speech never mentions, and it missed Ukraine, which it does. The entity model above found only what is in the text: Europe, Gaza, Russian, Ukraine. A prompted language model gives no counts, no list of words it used, and no reason for its answer, and it can answer differently each time. It can also produce things that are not in the text at all. Whatever a language model tells you about a text, check it against the text.

### Optional: few-shot, learning from examples in the prompt

All of this was zero-shot: no examples. We can also give the model a few examples to learn from. This is called *few-shot* or *in-context learning* (https://news.mit.edu/2023/large-language-models-in-context-learning-0207). In the chat format, the examples are simply earlier turns of the conversation: a question from the user, and the answer we want from the assistant. Run the next cell.

In [ ]:
#Keep cell
messages = [
    # the instruction
    {"role": "system", "content": "You are a helpful assistant. Classify the sentiment of the provided text. Respond with only one word: 'positive', 'negative', or 'neutral'."},
    # three examples: a question and the answer we want
    {"role": "user", "content": "Text: \"I absolutely love this speech! It's fantastic and works perfectly.\"\nSentiment:"},
    {"role": "assistant", "content": "positive"},
    {"role": "user", "content": "Text: \"The weather today is just okay, nothing special to report.\"\nSentiment:"},
    {"role": "assistant", "content": "neutral"},
    {"role": "user", "content": "Text: \"I'm very disappointed with the politician's vision. It was a terrible speech.\"\nSentiment:"},
    {"role": "assistant", "content": "negative"},
    # the real question
    {"role": "user", "content": f"Text: \"{text_sotu_europe}\"\nSentiment:"}
]
output = generator(messages, max_new_tokens=5, do_sample=False)
print(output[0]["generated_text"][-1]["content"])

Few-shot learning is easy to do and can work very well, with no training at all. Compare it with the fine-tuning above: which would you choose for hate speech, and why?

> **Background: instruction-tuned models** (optional reading)
>
> Qwen chat is an *instruction-tuned* model: a language model that has been trained further to follow instructions and to chat. Such models are becoming increasingly common, and they explain much of the excitement about language models. The details are complex, but it is enough for now to know that these models exist, and that they can do many of the tasks we used to train separate models for, from a prompt alone. Remember the *DeepLearning.AI* introduction to Python with AI? In the group exercise, you will query much larger models of this kind through their APIs.

> **Important:** Run the next cell to make an HTML file of your notebook for Canvas. It downloads automatically. First save the notebook (Ctrl+S, or ⌘+S on a Mac): the cell reads the notebook as it is on your screen, and if that does not work, it looks for your saved copy in Google Drive. If you are asked to allow access to your Google Drive, please confirm. Before you hand it in, open the HTML file and check that all your results are there. Its first line says when and how it was made.

In [ ]:
#Keep cell
# Makes an HTML file of this notebook and downloads it: hand in this file on Canvas
import os, importlib, urllib.request

# Always get the latest version of the helper (the old copy stays if the download fails)
try:
  urllib.request.urlretrieve('https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/make_html.py', 'make_html_download.py')
  os.replace('make_html_download.py', 'make_html.py')
except Exception as e:
  print('Could not download the latest make_html.py:', e)

import make_html
importlib.reload(make_html)
make_html.make_html("Week4_Workshop.ipynb")   # the name of your HTML file